<a href="https://colab.research.google.com/github/jhonmarkvillarosa11-commits/Laboratory-exam/blob/main/villarosa_Laboratory__exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Lab Exam: Classification, Complexity, and Interpretability
In this notebook, we will:
1. Preprocess a real-world tabular dataset (handling missing values, scaling, encoding).
2. Implement multiple supervised classification models (e.g., Logistic Regression, Decision Trees, Random Forests, Gradient Boosting).
3. Evaluate performance using metrics like Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
4. Analyze the trade-offs between model complexity, interpretability, and performance.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

print("Libraries successfully imported!")

Libraries successfully imported!


### Step 1: Scenario Dataset Generation (ConnectX Customer Churn)
We will generate a synthetic dataset named `connectx_churn.csv` featuring characteristics of typical telecom customer data (tenure, monthly charges, contract type, paperless billing, tech support, etc.).

In [7]:
import pandas as pd
import numpy as np

# Load the real-world dataset provided by the user
file_path = '/content/telecom_churn (1).csv'
df = pd.read_csv(file_path)

print("Dataset successfully loaded from:", file_path)
print(f"Shape: {df.shape}")
display(df.head())

Dataset successfully loaded from: /content/telecom_churn (1).csv
Shape: (3333, 21)


,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


### Step 3: Preprocessing Pipelines & Classifiers Setup
We will construct preprocessors using `ColumnTransformer` and prepare three supervised learning algorithms to highlight different trade-offs in complexity, interpretability, and performance.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Dynamically read current numerical and categorical columns from the actual splits
current_num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
current_cat_cols = X_train.select_dtypes(include=['object', 'bool']).columns.tolist()

# Define preprocessing steps
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessors with the correct active column names
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, current_num_cols),
        ('cat', categorical_transformer, current_cat_cols)
    ]
)

# Dictionary of classifiers to evaluate
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
}

print("Preprocessing pipelines updated successfully with columns:")
print("Numerical:", current_num_cols)
print("Categorical:", current_cat_cols)

Preprocessing pipelines updated successfully with columns:
Numerical: ['account length', 'area code', 'number vmail messages', 'total day minutes', 'total day calls', 'total day charge', 'total eve minutes', 'total eve calls', 'total eve charge', 'total night minutes', 'total night calls', 'total night charge', 'total intl minutes', 'total intl calls', 'total intl charge', 'customer service calls']
Categorical: ['state', 'phone number', 'international plan', 'voice mail plan']


### Step 4: Model Training and Evaluation
We will now train each model using our pipelines and compute standard metrics: Accuracy, F1-Score, and ROC-AUC to compare performance.

In [11]:
results = {}

for name, clf in classifiers.items():
    # Construct final pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    # Fit the pipeline
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1 = report['accuracy']
    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']

    results[name] = {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    }

# Display comparison dataframe
results_df = pd.DataFrame(results).T
display(results_df)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Logistic Regression,0.859070,0.831838,0.859070,0.859070,0.809387
Decision Tree,0.917541,0.913402,0.917541,0.917541,0.804350
Random Forest,0.854573,0.730295,0.854573,0.854573,0.862046


### Step 2: Data Preprocessing & Exploratory Analysis
Before building models, we must inspect and clean the dataset. For example, `TotalCharges` contains some spaces representation of missing records that must be converted to float and handled.

In [8]:
# Map target variable 'churn' (TRUE/FALSE) to 1/0
if 'churn' in df.columns:
    df['Churn'] = df['churn'].astype(int)
    df = df.drop(columns=['churn'])

# Clean feature names (remove spaces/dots if any) and inspect null values
df.columns = [col.strip() for col in df.columns]

# Check for missing values
missing_vals = df.isnull().sum()
print("Missing Values Per Column:")
print(missing_vals[missing_vals > 0])

# Drop unique identifier columns if present
identifiers = ['customerID', 'customer_id', 'ID', 'id', 'phone_number', 'phone_id']
cols_to_drop = [col for col in identifiers if col in df.columns]
df_clean = df.drop(columns=cols_to_drop)

# Impute missing numeric values with median, categorical with mode
for col in df_clean.columns:
    if col != 'Churn':
        if df_clean[col].dtype in ['int64', 'float64']:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        else:
            # If column is object/string, convert to string and handle empty spaces as NaN
            if df_clean[col].dtype == 'object':
                df_clean[col] = df_clean[col].astype(str).str.strip().replace('', np.nan)
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown')

# Separate features X and target y
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

# Identify numerical and categorical features
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"\nNumerical features: {num_cols}")
print(f"Categorical features: {cat_cols}")

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Missing Values Per Column:
Series([], dtype: int64)

Numerical features: ['account length', 'area code', 'number vmail messages', 'total day minutes', 'total day calls', 'total day charge', 'total eve minutes', 'total eve calls', 'total eve charge', 'total night minutes', 'total night calls', 'total night charge', 'total intl minutes', 'total intl calls', 'total intl charge', 'customer service calls']
Categorical features: ['state', 'phone number', 'international plan', 'voice mail plan']

Training set shape: (2666, 20)
Testing set shape: (667, 20)
